In [ ]:
# Healthcare_MDM – final source validation notebook

This notebook validates the packaged Python source without embedding temporary TEST_HCP production configuration or real API credentials.

In [ ]:
import sys
from pathlib import Path

# Update this path if the repository is cloned elsewhere in Databricks.
SRC_ROOT = Path("/Workspace/Users/naresh.mayari@gmail.com/Healthcare_Master_Data_Management/src")
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("Source path:", SRC_ROOT)


In [ ]:
# Import every project module.
from api import download_api, duplicate_jisb_records, process_jisb_response, process_orieo_response, sbc, transform_to_jisb, transform_to_orieo
from canonical import canonical
from core import data_io, logging_utils, runtime_config
from dq import data_quality
from ingestion import src_to_raw_ingestion
from mdm import mdm_ingress, mdm_egress
from standardization import standardization, standardization_function

print("All project modules imported successfully.")


In [ ]:
# Confirm the centralized runtime configuration resolves the project schemas.
print("catalog:", runtime_config.catalog)
print("raw:", runtime_config.raw_schema)
print("landing:", runtime_config.lnd_schema)
print("staging:", runtime_config.stg_schema)
print("mdm:", runtime_config.publish_schema)
print("util:", runtime_config.util_schema)
print("DQ config:", runtime_config.dqm_config_tbl)


In [ ]:
# Validate that the project DQ implementation contains the 24 workbook rules.
print("Workbook DQ rules:", len(data_quality.DQ_RULES))
assert len(data_quality.DQ_RULES) == 24
print("DQ rule set validation: PASS")


In [ ]:
# Pure transformation smoke test using non-sensitive mock input only.
mock_hcp = {
    "hcp": {
        "firstName": "John",
        "middleName": "A",
        "lastName": "Smith",
    },
    "address": {
        "primary": "10 Main Street",
        "countryCode": "NL",
        "city": "Amsterdam",
        "longPostalCode": "1011AB",
        "type": "Primary",
    },
}

jisb = transform_to_jisb.transform_to_jisb(mock_hcp)
print("JISB transformation produced keys:", list(jisb.keys()))
assert isinstance(jisb, dict)
print("JISB transform smoke test: PASS")


In [ ]:
# Show the batch gating functions that control pipeline progression.
for module in ["stdz", "canonical", "dq", "ingress", "egress"]:
    print(module, "=>", runtime_config.get_batch_status_filter(module, "TEST"))
print("Batch gating validation: PASS")


In [ ]:
# IMPORTANT: live pipeline execution is intentionally opt-in.
# Before running ingestion/API/MDM stages, configure the real control rows,
# physical tables, secret references, and approved runtime mappings.
# Do not paste API passwords or Informatica credentials into this notebook.
print("Final validation notebook loaded. Live execution requires environment-specific configuration.")
